# Independent held-out DAS registration checkpoint

This advisor-facing notebook reads only compact preregistration ledgers. It verifies that all 12 padded intervals have manifest coverage and that no HDF5 header/dataset, network/catalog candidate time, or family label was opened. It does **not** run the detector or make an extension claim.

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'config' / 'heldout_das_replay.json').is_file()
)
REGISTRATION = ROOT / 'outputs' / 'heldout_v2' / 'registration'

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

config_path = ROOT / 'config' / 'heldout_das_replay.json'
status_path = REGISTRATION / 'das_registration_status.json'
manifest_path = REGISTRATION / 'das_manifest_selection.csv'
interval_path = REGISTRATION / 'das_interval_selection.csv'
config = json.loads(config_path.read_text())
status = json.loads(status_path.read_text())
manifest = pd.read_csv(manifest_path)
intervals = pd.read_csv(interval_path)

assert status['status'] == 'PASS'
assert status['heldout_DAS_config_sha256'] == sha256(config_path)
assert status['manifest_selection_sha256'] == sha256(manifest_path)
assert status['interval_selection_sha256'] == sha256(interval_path)
assert len(intervals) == 12 and len(manifest) == 738
assert intervals['status'].eq('PASS').all()
assert manifest['path'].nunique() == len(manifest)
for field in ['hdf5_file_opened', 'hdf5_header_opened', 'hdf5_dataset_opened']:
    assert manifest[field].astype(str).str.lower().eq('false').all()
for field in [
    'heldout_DAS_HDF5_files_opened',
    'heldout_DAS_HDF5_headers_opened',
    'heldout_DAS_HDF5_datasets_opened',
    'network_candidate_table_rows_opened',
    'catalog_association_table_rows_opened',
    'network_or_catalog_candidate_time_fields_read',
    'heldout_family_label_rows_opened',
]:
    assert status[field] == 0
print('PASS: hashes, 12/12 coverage, and zero-access guards verify')

In [ ]:
summary = pd.DataFrame({
    'metric': [
        'registered intervals',
        'unique selected HDF5 paths',
        'registered bytes (GB)',
        'maximum manifest gap (s)',
        'HDF5 headers/datasets opened',
        'network/catalog candidate times read',
        'candidate rows materialized',
    ],
    'value': [
        len(intervals),
        manifest['path'].nunique(),
        manifest['size_bytes'].sum() / 1e9,
        intervals['maximum_manifest_gap_s'].max(),
        status['heldout_DAS_HDF5_headers_opened'] + status['heldout_DAS_HDF5_datasets_opened'],
        status['network_or_catalog_candidate_time_fields_read'],
        status['base_v1_candidate_rows_materialized'] + status['v2_candidate_rows_materialized'],
    ],
})
display(summary)
print('Current gate:', status['heldout_DAS_waveform_access_gate'])
print('Next gate:', status['next_stage_gate'])

In [ ]:
plot_rows = intervals.assign(registered_GB=intervals['selected_total_size_bytes'] / 1e9)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(plot_rows['interval_id'], plot_rows['selected_manifest_record_count'])
axes[0].set_ylabel('Selected files')
axes[0].set_title('Manifest files per padded hour')
axes[1].bar(plot_rows['interval_id'], plot_rows['registered_GB'], color='tab:orange')
axes[1].set_ylabel('Registered input (GB)')
axes[1].set_title('Read-only input footprint')
for ax in axes:
    ax.tick_params(axis='x', rotation=60)
    ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
# Display-only advisor controls: changing these never writes a product.
DISPLAY_INTERVAL = 'heldout_01'  # use 'all' to show every interval
MAX_FILE_ROWS = 12
SORT_FIELD = 'manifest_start_utc'

shown_intervals = intervals if DISPLAY_INTERVAL == 'all' else intervals[intervals['interval_id'] == DISPLAY_INTERVAL]
shown_files = manifest if DISPLAY_INTERVAL == 'all' else manifest[manifest['interval_id'] == DISPLAY_INTERVAL]
display(shown_intervals)
display(shown_files.sort_values(SORT_FIELD).head(MAX_FILE_ROWS)[[
    'interval_id', 'interval_file_index', 'manifest_start_utc',
    'manifest_end_utc', 'size_bytes', 'file_exists'
]])

## Decision and next gate

Registration is complete, but waveform access remains **STOP** until a fail-closed runner and its tests are committed, pushed, and remotely released. That runner must scan every sample in all 12 intervals, retain the complete base-v1 threshold table, freeze the four-of-ten v2 subset, and checksum both before any network/catalog-time comparison. Only the later comparison can test whether DAS adds independently validated events beyond the full network union.